In [ ]:
import os
from pathlib import Path
import pandas as pd
import psycopg
from dotenv import load_dotenv

ENV_FILE = Path.cwd().parent / "setenv_STAGING.sh"
load_dotenv(ENV_FILE)

conn = psycopg.connect(
    host=os.environ["ADYN_POSTGRES_HOST"],
    port=os.environ.get("ADYN_POSTGRES_PORT", "5432"),
    dbname=os.environ["ADYN_POSTGRES_DB"],
    user=os.environ["ADYN_POSTGRES_USER"],
    password=os.environ["ADYN_POSTGRES_PASSWORD"],
)
conn.autocommit = True

def q(sql, params=None):
    """Run a SQL query and return the result as a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [c.name for c in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

In [ ]:
# 1. how many patients have both lab results and prescriptions
print(q("""
with h as (select distinct customer_id from hormonevalues),
     g as (select distinct customer_id from genotypevalues),
     r as (select distinct customer_id from rxorders)
select (select count(*) from h) as hormones,
       (select count(*) from g) as genotypes,
       (select count(*) from r) as prescriptions,
       (select count(*) from h join r using (customer_id)) as hormones_and_rx,
       (select count(*) from g join r using (customer_id)) as genotypes_and_rx,
       (select count(*) from h join g using (customer_id) join r using (customer_id)) as all_three
""").to_string())

# which prescription products exist, and what statuses their orders have
print(q("""
select sku, count(*) as orders, count(distinct customer_id) as customers,
       string_agg(distinct status, ', ') as statuses
from rxorders
group by sku
order by orders desc
""").to_string())

1. How many patients have both lab results and prescriptions?

42
 - Of 45 patients with any prescription order, 42 also have both hormone and genetic results
 - 192 patients have hormone results and 194 have genetic results. Every prescription is birth control, across about 10 products, with Daysee, Junel Fe and Falmina the most common.

In [ ]:
# 2. hormone coverage and distributions
print(q("""
select h.name, count(*) as n_values, count(distinct hv.customer_id) as n_customers,
       min(hv.hormone_value) as min,
       percentile_cont(0.5) within group (order by hv.hormone_value) as median,
       max(hv.hormone_value) as max,
       round(avg((hv.hormone_value < hv.ref_low)::int), 2) as share_low,
       round(avg((hv.hormone_value > hv.ref_high)::int), 2) as share_high
from hormonevalues hv join hormones h on h.id = hv.hormone_id
group by h.name order by n_customers desc
""").to_string())

# checking reference ranges and placeholder values
print(q("""
select h.name,
       count(*) filter (where hv.ref_low is null or hv.ref_high is null) as missing_ref,
       min(hv.ref_low) as min_ref_low, max(hv.ref_low) as max_ref_low,
       min(hv.ref_high) as min_ref_high, max(hv.ref_high) as max_ref_high,
       count(*) filter (where hv.hormone_value < 0) as negative,
       count(*) filter (where hv.hormone_value = 'Infinity') as infinite
from hormonevalues hv join hormones h on h.id = hv.hormone_id
group by h.name
""").to_string())

2. Hormone coverage and distributions


- All 192 hormone patients have all 6 hormones (AMH, DHEA-S, E2, FSH, LH, SHBG), each measured about twice (373 values per hormone). 
- Medians are AMH 3.07, DHEA-S 129, E2 298.9, FSH 67.0, LH 28.9 and SHBG 69.3. 
- Almost every value falls inside its reference range (1% or less outside), because each patient has their own range. 
- There are 4 bad values (3 negative, 1 infinite), which were excluded.

In [ ]:
# 3. which genetic scores exist in pipeline output (top-level JSON keys)

# top-level keys in pipeline_output
print(q("""
select key, jsonb_typeof(value) as type, count(*) as n_reports
from resultvalues, jsonb_each(pipeline_output::jsonb)
where pipeline_output is not null
group by key, type
order by n_reports desc, key
""").to_string())

# which genetic scores exist (keys inside "genetics")
print(q("""
select key, jsonb_typeof(value) as type, count(*) as n_reports
from resultvalues, jsonb_each(pipeline_output::jsonb -> 'genetics')
group by key, type order by n_reports desc, key
""").to_string())

# fields stored inside each genetic score
print(q("""
select g.key as score, k.key as field, jsonb_typeof(k.value) as type, count(*) as n_reports
from resultvalues,
     jsonb_each(pipeline_output::jsonb -> 'genetics') g,
     jsonb_each(g.value) k
group by score, field, type
order by score, n_reports desc, field
""").to_string())


3. Which genetic scores already exist in pipeline output?

- depprs and vtesnp, both present in all 311 reports
    - not sure what each stand for
    - depprs has prs (raw score) + zscore (standardized, 0 = average; mean -0.1, SD 1.0, range -2.5 to 3.5)
    - vtesnp contains:
        - F5_Leiden (Factor V Leiden, rs6025)
        - F2_Prothrombin (Prothrombin, rs1799963)
        - looks like theyre both blood clotting factor/proteins but not sure abt actual meaning, need to look into it

In [ ]:
# 4. prescriptions tried by hormone level (patients split into thirds per hormone)
# Hormone values, without placeholders; repeat tests get averaged below
hv = q("""
select hv.customer_id, h.name as hormone, hv.hormone_value::float as value
from hormonevalues hv join hormones h on h.id = hv.hormone_id
where hv.hormone_value >= 0 and hv.hormone_value <> 'Infinity'
""")
# distinct birth control products per patient (skips consult charges, EXTERNAL and FREE)
rx = q("""
select customer_id, count(distinct sku) as n_rx_tried
from rxorders
where sku like 'AD-BC-%' and sku not in ('AD-BC-EXTERNAL', 'AD-BC-FREE0000')
group by customer_id
""")

# one row per patient, one column per hormone & how many products they tried
per = hv.groupby(["customer_id", "hormone"])["value"].mean().unstack()
df = per.join(rx.set_index("customer_id"), how="left").fillna({"n_rx_tried": 0})

rows = []
for h in per.columns:
    third = pd.qcut(df[h], 3, labels=["low", "middle", "high"])
    s = df.groupby(third, observed=True)["n_rx_tried"].agg(
        patients="size",
        share_with_rx=lambda x: (x > 0).mean(),
        avg_rx_tried_if_any=lambda x: x[x > 0].mean(),
    )
    s.insert(0, "hormone", h)
    rows.append(s.reset_index(names="third"))
print(pd.concat(rows).round(2).to_string(index=False))

4. Prescriptions tried by hormone range

No clear pattern so far. 
- For each hormone, split patients into thirds (low, middle and high)
- The share who tried any birth control stayed between 13% and 32% across the thirds, and the average number of products tried ranged from 1.2 to 3.1.
-  Each group has only about 10 to 20 patients with prescriptions -- differences could be by chance

In [ ]:
# 5. prescriptions tried by genetic score
d = gen.merge(rx, on="customer_id", how="left").fillna({"n_rx_tried": 0})
d["depression_risk"] = pd.qcut(d["dep_zscore"], 3, labels=["low", "middle", "high"])
d["clot_carrier"] = (d["f5_leiden"] > 0) | (d["f2_prothrombin"] > 0)

for col in ["depression_risk", "clot_carrier"]:
    print(d.groupby(col, observed=True)["n_rx_tried"].agg(
        patients="size",
        n_with_rx=lambda x: (x > 0).sum(),
        share_with_rx=lambda x: (x > 0).mean(),
        avg_rx_tried_if_any=lambda x: x[x > 0].mean(),
    ).round(2).to_string(), "\n")

5. prescriptions tried by genetic score

No reliable pattern yet. 
- Patients in the high depression-risk third had a prescription more often: 36%, compared with 24% for the low third and 20% for the middle third.
    - only 18 vs 12 vs 10 patients -- could be js by chance

In [ ]:
#6.  hormone impact on ordering/prescription
from scipy import stats

first = q("""
select distinct on (customer_id) customer_id, sku as first_product
from rxorders
where sku like 'AD-BC-%' and sku not in ('AD-BC-EXTERNAL', 'AD-BC-FREE0000')
order by customer_id, date_order_placed
""").set_index("customer_id")
dd = df.join(first, how="left")
any_rx = dd["n_rx_tried"] > 0

results = []
for h in per.columns:
    vals = dd[h].dropna()
    has, no = vals[any_rx[vals.index]], vals[~any_rx[vals.index]]
    p_any = stats.mannwhitneyu(has, no).pvalue
    rho, p_count = stats.spearmanr(dd.loc[has.index, h], dd.loc[has.index, "n_rx_tried"])
    groups = [g[h].dropna() for _, g in dd.dropna(subset=["first_product"]).groupby("first_product")
              if g[h].notna().sum() >= 3]
    p_product = stats.kruskal(*groups).pvalue
    results.append({"hormone": h, "median_with_rx": has.median(), "median_no_rx": no.median(),
                    "p_any_rx": p_any, "rho_n_tried": rho, "p_n_tried": p_count,
                    "p_first_product": p_product})

print(pd.DataFrame(results).round(3).to_string(index=False))


6. hormone + ordering/prescription

- none of the hormones has a statistically significant effect on prescriptions
- all 18 tests are above the 0.0028 cutoff
    - covers having any prescription & how many products were tried

closest result: LH
    - patients with a prescription had lower LH (median 24.1 vs 30.6, p = 0.013) 
        -would pass the usual 0.05 cutoff, but not after correcting for running 18 tests

number of products tried: the correlations are weak, between -0.10 and 0.22, and none are significant


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BLUE, INK, MUTED = "#2a78d6", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": MUTED, "xtick.color": MUTED,
    "ytick.color": MUTED, "axes.titlesize": 11, "axes.titlecolor": INK,
})

# Chart 1: distribution of each hormone (placeholders already removed in hv)
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (name, g) in zip(axes.flat, hv.groupby("hormone")):
    ax.hist(g["value"], bins=25, color=BLUE, edgecolor="white", linewidth=1)
    ax.set_title(name)
    ax.set_ylabel("results")
fig.suptitle("Hormone distributions", color=INK)
fig.tight_layout()
plt.show()

# Chart 2: patients per birth control product
prod = q("""
select sku, count(distinct customer_id) as patients
from rxorders
where sku like 'AD-BC-%' and sku not in ('AD-BC-EXTERNAL', 'AD-BC-FREE0000')
group by sku order by patients
""")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(prod["sku"], prod["patients"], color=BLUE, height=0.6)
for y, v in enumerate(prod["patients"]):
    ax.text(v + 0.3, y, str(v), va="center", color=MUTED, fontsize=9)
ax.set_xlabel("patients")
ax.set_title("Patients per birth control product", loc="left")
fig.tight_layout()
plt.show()

# Chart 3: share of patients with any prescription, by hormone third
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharey=True)
for ax, h in zip(axes.flat, per.columns):
    third = pd.qcut(df[h], 3, labels=["low", "middle", "high"])
    g = df.groupby(third, observed=True)["n_rx_tried"]
    share, n_rx = g.apply(lambda x: (x > 0).mean()), g.apply(lambda x: (x > 0).sum())
    ax.bar(share.index.astype(str), share.values, color=BLUE, width=0.6)
    for x, (s, n) in enumerate(zip(share, n_rx)):
        ax.text(x, s + 0.01, f"{s:.0%}\n(n={n})", ha="center", color=MUTED, fontsize=9)
    ax.set_title(h)
    ax.set_ylim(0, 0.5)
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
axes[0, 0].set_ylabel("share with a prescription")
axes[1, 0].set_ylabel("share with a prescription")
fig.suptitle("Share of patients with any birth control prescription, by hormone level (thirds)", color=INK)
fig.tight_layout()
plt.show()